In [ ]:
%%time 

# autoload external python modules if they changed
%load_ext autoreload
%autoreload 2

# show plots inline in a notebook (no extra spaces/characters are allowed after 'inline')
%matplotlib inline

# add ../funcs to the current path
import sys, os
sys.path.append(os.path.join(os.getcwd(), "../funcs")) 

# Change Matplotlib's Default DPI
import matplotlib as mpl
mpl.rcParams['figure.dpi'] = 250

In [ ]:
%%time

# read data from netcdf files, save them to 'datasets'
from load_inv_bkg_ana import load_inv_bkg_ana
files = {
    "inv": "../data/samples/mpasjedi/invariant.nc",
    "bkg": "../data/samples/mpasjedi/bkg.nc",
    "ana": "../data/samples/mpasjedi/ana.nc",
}
datasets = {}
load_inv_bkg_ana(files, datasets)

In [ ]:
%%time
from base import source
import os
expdir = "/lfs6/BMC/wrfruc/MPAS_dev4/exp/v2.0.9/rrfsdet"
cdate = "2025032216"

source(f"{expdir}/exp.setup")
NET = os.getenv("NET")
RUN = NET
WGF = os.getenv("WGF")
TAG = os.getenv("TAG")
COMROOT = os.getenv("COMROOT")
DATAROOT = os.getenv("DATAROOT")
with open(f"{expdir}/VERSION", "r") as file:
    VERSION = file.readline().strip()
#print(NET, RUN, WGF, TAG, VERSION, COMROOT, DATAROOT)

# find the correct invariant.nc
jedivar_log = f"{COMROOT}/{NET}/{VERSION}/logs/{RUN}.{cdate[:8]}/{cdate[8:10]}/{WGF}/{RUN}_jedivar_{TAG}_{cdate}.log"
end_str="./invariant.nc"
with open(f"{jedivar_log}", "r") as file:
    for line in file:
        line = line.strip()
        if line.endswith(end_str):            
            inv_file = line[:-len(end_str)].split(":",1)[1].strip()[len("ln -snf"):].strip()
            break
#print(inv_file)

# find the background file from the prep_ic log file
prep_ic_log = f"{COMROOT}/{NET}/{VERSION}/logs/{RUN}.{cdate[:8]}/{cdate[8:10]}/{WGF}/{RUN}_prep_ic_{TAG}_{cdate}.log"
start_str = "warm start from"
with open(f"{prep_ic_log}", "r") as file:
    for line in file:
        if line.startswith(start_str):
            bkg_file = line[len(start_str):].strip()
            break
#print(bkg_file)

# find the analysis file from the UMBRELLA_PREP_IC
ana_file = f"{DATAROOT}/{cdate[:8]}/{RUN}_prep_ic_{cdate[8:10]}_{VERSION}/{WGF}/mpasin.nc"
# print(ana_file)

# read data from netcdf files, save them to 'datasets'
from load_inv_bkg_ana import load_inv_bkg_ana
files = {
    "inv": inv_file,
    "bkg": bkg_file,
    "ana": ana_file,
}
datasets = {}
load_inv_bkg_ana(files, datasets)

In [ ]:
%%time

# make plots
from contour_increment import contour_increment
parms={    
    "plot_box_width": 80.0,
    "plot_box_height": 40.0,
    "cen_lat": 34.5,
    "cen_lon": -97.5,
    "convert_theta_to_t": True
}
parms['ilevel'] = 2
contour_increment(datasets, parms)
#contour_increment(datasets, parms, f"L{parms['ilevel']}")
parms['ilevel'] = 20
contour_increment(datasets, parms, f"L{parms['ilevel']}")
parms['ilevel'] = 40
contour_increment(datasets, parms, f"L{parms['ilevel']}")